# When should the pump stay in alarm?
**Intro to ML · Time series laboratory · 90 minutes · STUDENT VERSION**

One real industrial task, several representations, one frozen held-out evaluation.
We compare robust statistics, PCA, Isolation Forest, score fusion and a recorded
**Chronos-2-small** forecasting detector on SKAB pump measurements.

**Required hardware:** free Colab CPU, standard RAM. No account token, API key,
GPU or foundation-model download is required for the assessed route.
Fresh classical fitting runs here. Foundation forecasts are explicitly recorded
outputs of the exact causal protocol documented below. An optional live extension
appears at the end.

**Your work:** four small code tasks, predictions, failure analysis and one final decision.
Prepared for Intro to ML with the instructor and OpenAI Codex.
The instructor retains responsibility for accuracy and assessment.


## The route
| Minutes | Question |
|---:|---|
| 0–18 | What is the fault state, and what information is available now? |
| 18–30 | Can a normal relationship fail while individual values look plausible? |
| 30–45 | Does history help the same Isolation Forest? |
| 45–57 | What does a detector score mean on a different scale? |
| 57–68 | Does combining three detectors help? |
| 68–78 | Does forecasting error cover the whole fault state? |
| 78–90 | Freeze, reveal held-out runs, and defend a decision. |

Run this notebook **section by section**. The four task functions are implemented below. A fresh top-to-bottom run should work
when the supplied assets and dependencies are available. Supplied plotting, fitting and evaluation helpers
are labelled infrastructure. The four algorithms are absent from those helpers.


## Predict before measuring
Write a direction and a mechanism, not an invented numerical score. Keep these
predictions after seeing the results: changing your mind is part of the experiment.


In [ ]:
PREDICTIONS = {
    "history": (
        "Causal history features may help the Isolation Forest recognize a sustained "
        "fault state. Rolling windows may delay alarms near transitions and can add "
        "false alarms when a healthy regime changes."
    ),
    "fusion": (
        "Equal-weight fusion may improve coverage when detectors catch different "
        "parts of a fault. Correlated scores can add little, while a strong isolated "
        "score can be diluted by the mean."
    ),
    "forecasting": (
        "Forecast residuals may be strongest at fault onset, then weaken after the "
        "new state enters the context. The matched persistence forecast may remain "
        "competitive on slowly changing signals."
    ),
}
PREDICTIONS


## Reproducible CPU setup
Colab already provides the scientific libraries used here. Keep that working
stack: the next cell checks and reports its versions, installing a reference
version only when a package is absent. The recorded forecast environment is
documented separately. If installing a missing package changes an already
loaded dependency, restart once, then rerun from the top.


In [ ]:
#@title Check CPU dependencies { display-mode: "form" }
import importlib.metadata
import os
import subprocess
import sys
CORE_PINS = ["numpy==2.2.6", "pandas==2.2.3", "scipy==1.15.2", "scikit-learn==1.6.1", "matplotlib==3.10.7", "joblib==1.6.0"]
MODULES = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "scikit-learn": "sklearn", "matplotlib": "matplotlib", "joblib": "joblib"}
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
before = {name: installed_version(name) for name in MODULES}
loaded_before = {name for name, module in MODULES.items() if module in sys.modules}
missing_pins = [pin for pin in CORE_PINS if before[pin.split("==")[0]] is None]
if missing_pins and os.environ.get("SKAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pins])
versions = {name: installed_version(name) for name in MODULES}
changed_loaded = [name for name in loaded_before if before[name] != versions[name]]
print({"policy": "Keep preinstalled packages; install only missing dependencies", "versions": versions})
if changed_loaded:
    raise RuntimeError("Dependencies changed while loaded: " + ", ".join(changed_loaded) + ". Restart the session once, then Run all.")


### Supplied, inspectable assets
The next folded cell contains a compressed package of the twenty pinned public
CSVs, readable helpers and **actual recorded forecast arrays**. It does not contain
model weights, credentials, instructor solutions or invented model outputs.
Expand it to inspect the bootstrap. Every package member is checked by SHA-256.
Existing unrelated files are never overwritten; results use a fresh attempt directory.


In [ ]:
#@title Verify and unpack supplied data + recorded forecasts { display-mode: "form" }
import base64, hashlib, io, json, uuid, zipfile
from pathlib import Path
PACKAGE_B64 = "..."
PACKAGE_SHA256 = "25cc9412d42b95d92a435392844acedea3dccb5a39021cc0477b6ca88693cfee"
payload = base64.b64decode(PACKAGE_B64)
assert hashlib.sha256(payload).hexdigest() == PACKAGE_SHA256
BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "work"
ASSETS = BASE / ("skab_lab_assets_" + PACKAGE_SHA256[:12])
ASSETS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    manifest = json.loads(archive.read("package_manifest.json"))
    for name, expected in manifest.items():
        member = Path(name)
        assert not member.is_absolute() and ".." not in member.parts
        content = archive.read(name)
        assert len(content) == expected["bytes"] and hashlib.sha256(content).hexdigest() == expected["sha256"]
        target = ASSETS / member
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert hashlib.sha256(target.read_bytes()).hexdigest() == expected["sha256"]
        else:
            target.write_bytes(content)
print({"verified_assets":len(manifest),"package_bytes":len(payload),"assets":str(ASSETS)})


In [ ]:
import importlib
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
# Reload the exact verified package, even after opening a newer notebook in this kernel.
for module_name in ['skab_lab', 'lab_runtime', 'forecast_diagnostics']:
    sys.modules.pop(module_name, None)
if str(ASSETS) in sys.path:
    sys.path.remove(str(ASSETS))
sys.path.insert(0, str(ASSETS))
importlib.invalidate_caches()
import skab_lab as core
from skab_lab import CHANNELS, CONFIG
from lab_runtime import LabRuntime, DETECTORS, compact_report
from forecast_diagnostics import forecast_quality_validation
for task_name in ['pca_q', 'window_features', 'ecdf_transform', 'fuse_scores']:
    if hasattr(core, task_name):
        delattr(core, task_name)  # A new attempt must not inherit an old solution.
# Re-running setup starts a separately identified attempt, never overwrites a frozen one.
ATTEMPT = BASE / 'skab_lab_runs' / uuid.uuid4().hex
SESSION = LabRuntime(ASSETS, ATTEMPT)
pd.DataFrame({"item": ["SKAB revision", "forecast model", "forecast revision", "forecast evidence", "current device"],
              "value": [CONFIG["skab_commit"], SESSION.forecast_receipt["model"], SESSION.forecast_receipt["revision"], SESSION.forecast_receipt["status"], "CPU — classical fitting runs now"]})


## Split whole runs before creating windows
**Fit:** valve1/0–3, valve2/0. **Normal calibration:** valve1/4–7, valve2/1.
**Validation:** valve1/8–11, valve2/2. **Held-out:** valve1/12–15, valve2/3.

Detectors and the robust scaler see only healthy prefixes of **fit runs**.
Score CDFs use healthy prefixes of separate **calibration runs**. Validation
labels select the method and thresholds. Held-out runs select nothing.
This is same-rig run-held-out evaluation; it does not test new machines or unseen fault families.
Calendar times overlap across the valve families: this is not a forward-in-calendar-time deployment simulation.

Timestamps are put on a one-second grid. At most two missing seconds are filled
**from the past**, only to supply context. Longer gaps reset the segment.
Every method scores only actually observed rows after a common 128-second warmup.
No window, forecast context or interpolation crosses a reset.


In [ ]:
SESSION.split_table()


### Read the physical experiment
The pump has eight measurement channels. Below are vibration RMS, motor current
and flow rate. Find a change, a sustained regime and a recovery. The vibration
channels are already RMS summaries: they are not raw high-frequency accelerometer waveforms.


In [ ]:
SESSION.plot_observations('valve1/8')


### State, onset and transition are different targets
The shaded `anomaly=1` interval is our primary target: **keep the alarm active
while the installation remains in the annotated faulty state**. SKAB commonly
has four `changepoint` marks for a run: entering a transition, settling into
the fault, beginning recovery, and returning to normal.

Our secondary onset metric asks whether the first state onset is detected
within 60 seconds, and reports delay only for detections. One early hit does
not turn the rest of the interval into correct predictions.


In [ ]:
validation_labels = np.concatenate([SESSION.frames[r].loc[SESSION.frames[r].eligible, 'anomaly'].to_numpy(dtype=int)
                                    for r in core.SPLITS['validation']])
pd.DataFrame([{"detector": "always normal", "accuracy": float((validation_labels == 0).mean()),
               "state_recall": 0.0, "state_F1": 0.0, "normal_alarm_fraction": 0.0}])


## A simple score worth beating
Fit one median $m_j$ and robust scale $a_j$ for each channel on healthy fit data.
Our supplied baseline is $s(t)=\max_j |(x_j(t)-m_j)/a_j|$.
A zero MAD uses the fit-set standard deviation as an explicit fallback.
The maximum is interpretable: one unusual channel can trigger the detector.
It cannot directly identify a changed relationship between otherwise plausible values.


In [ ]:
pd.DataFrame({'channel': CHANNELS, 'normal_fit_median': SESSION.center, 'normal_fit_scale': SESSION.scale})


## Task 1/4 — distance from normal relationships
PCA learns a three-dimensional subspace from the eight standardized healthy
measurements. For an observation $z$ and its reconstruction $\hat z$, compute
$Q=\sum_j(z_j-\hat z_j)^2$. Return **one score per row**, not one scalar for the batch.
A large shift inside the retained subspace can still have small Q.


In [ ]:
def pca_q(z: np.ndarray, reconstruction: np.ndarray) -> np.ndarray:
    """Return the squared reconstruction residual for each observation."""
    return np.square(z - reconstruction).sum(axis=1)


In [ ]:
q_example = pca_q(np.array([[1., 2.], [3., 4.]]), np.array([[1., 0.], [3., 4.]]))
assert q_example.shape == (2,) and np.allclose(q_example, [4., 0.])
assert np.allclose(pca_q(np.array([[1., -2., 3.], [0., 0., 0.], [2., 2., 2.]]), np.zeros((3, 3))), [14., 0., 12.])
core.pca_q = pca_q
pd.DataFrame({'example': ['residual [0, 2]', 'exact reconstruction'], 'Q': q_example})


In [ ]:
SESSION.fit_pca()


**Read the result:** explained variance describes normal training geometry.
It is not anomaly-detection accuracy. What failure would Q miss if the faulty
observation moved along one of these retained normal directions?


## Task 2/4 — same forest, different representation
Produce 48 causal features: 8 current standardized values, 8 first differences,
and 8 means + 8 standard deviations at each of **30 and 120 seconds**.
`std` uses `ddof=0`. A full window is required; early NaNs are removed by the
shared 128-second warmup. Group by segment so history cannot cross a run/gap.

`DataFrame.rolling(window, min_periods=window)` is the supplied API clue.
Predict a trade-off before running: persistent state evidence versus transition timing.


In [ ]:
def window_features(frame: pd.DataFrame, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Build current, difference, and causal rolling features per segment."""
    z = pd.DataFrame((frame[CHANNELS].to_numpy() - center) / scale, index=frame.index)
    result = []
    for _, block in z.groupby(frame.segment_id.to_numpy(), sort=False):
        fields = [block.to_numpy(), block.diff().to_numpy()]
        for window in CONFIG["windows_seconds"]:
            rolling = block.rolling(window=int(window), min_periods=int(window))
            fields.extend((rolling.mean().to_numpy(), rolling.std(ddof=0).to_numpy()))
        result.append(np.concatenate(fields, axis=1))
    return np.concatenate(result, axis=0)


In [ ]:
probe = SESSION.frames['valve1/8'].iloc[:350].copy()
feature_probe = window_features(probe, SESSION.center, SESSION.scale)
changed_future = probe.copy()
changed_future.iloc[250:, changed_future.columns.get_indexer(CHANNELS)] += 1000
changed_features = window_features(changed_future, SESSION.center, SESSION.scale)
assert feature_probe.shape == (len(probe), 48)
assert np.allclose(feature_probe[:250], changed_features[:250], equal_nan=True), 'Future values changed past features'
assert np.isfinite(feature_probe[probe.eligible]).all()
# Independent analytic fixture: ramps have a known mean and population std.
slopes = np.arange(1., 9.)
synthetic = pd.DataFrame(np.tile(np.arange(160.), 2)[:, None] * slopes
                         + np.repeat([0., 1000.], 160)[:, None], columns=CHANNELS)
synthetic['segment_id'] = np.repeat([0, 1], 160)
fixture = window_features(synthetic, np.zeros(8), np.ones(8))
expected = np.concatenate([149 * slopes, slopes, 134.5 * slopes,
                           np.sqrt(899 / 12) * slopes, 89.5 * slopes,
                           np.sqrt(14399 / 12) * slopes])
assert np.allclose(fixture[149], expected), 'Check current/difference/mean/std, column order and ddof=0'
assert np.isnan(fixture[160, 8:]).all(), 'Every history feature must reset at a new segment'
assert np.isnan(fixture[278, 32:]).all(), 'A full 120-position window is required'
assert np.allclose(fixture[279, 32:40], 1000 + 59.5 * slopes), 'No history may cross the segment boundary'
core.window_features = window_features
{'feature_shape': feature_probe.shape, 'future_perturbation_affects_past': False,
 'eligible_rows_finite': True, 'analytic_rolling_values': True, 'population_std': True, 'segment_reset': True}


In [ ]:
SESSION.fit_forests()


### Fresh classical scores; recorded forecast scores
The next stage computes classical scores now. It joins the supplied forecast
cache by exact `(run_id, datetime)` keys. Missing or duplicate keys raise an error.
The cache used context=128, horizon=16 and stride=16; all eight sensors are a
multivariate target and every forecast uses only earlier observations.
Calibration uses its healthy prefix only. Held-out signals are not scored or shown yet.


In [ ]:
SESSION.score_development()


In [ ]:
SESSION.plot_calibration_tails()


## Task 3/4 — compare scores without trusting their units
A raw sum can mostly reproduce the detector with the largest numerical scale.
Map each score against its own **frozen healthy calibration distribution**:
$u(s)=(1+\#\{c_i\le s\})/(n+2)$.
Sort the calibration array, then use `np.searchsorted(..., side="right")`.
Do not rank the validation/test scores against themselves.
First inspect the raw healthy tails above: calibration and validation normal
observations can occupy different score ranges.


In [ ]:
def ecdf_transform(scores: np.ndarray, calibration: np.ndarray) -> np.ndarray:
    """Map scores to smoothed ranks using a frozen calibration reference."""
    scores = np.asarray(scores, dtype=float)
    calibration = np.asarray(calibration, dtype=float)
    if calibration.ndim != 1 or calibration.size == 0 or not np.isfinite(calibration).all():
        raise ValueError("calibration must be a nonempty finite one-dimensional array")
    if scores.ndim != 1 or not np.isfinite(scores).all():
        raise ValueError("scores must be a finite one-dimensional array")
    reference = np.sort(calibration)
    ranks = np.searchsorted(reference, scores, side="right")
    return (1 + ranks) / (len(reference) + 2)


In [ ]:
example_ranks = ecdf_transform(np.array([0., 1., 2., 3.]), np.array([2., 1., 2.]))
assert np.allclose(example_ranks, [.2, .4, .8, .8])
for invalid_reference in [np.array([]), np.array([np.nan]), np.array([np.inf])]:
    try:
        ecdf_transform(np.array([1.]), invalid_reference)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject empty/nonfinite calibration references')
assert np.allclose(ecdf_transform(np.array([2.]), np.array([2., 1., 2.])), [.8]), 'Ranks depend only on the frozen reference'
core.ecdf_transform = ecdf_transform
pd.DataFrame({'score': [0., 1., 2., 3.], 'normal_reference_rank': example_ranks})


In [ ]:
SESSION.normalize_scores()


**Read the saturation column.** Every value above the largest calibration score
receives the same empirical rank. A small reference set cannot resolve an arbitrarily
rare tail. A rank of 0.99 is not a 99% posterior probability of a fault, and normal
regime shift can move many healthy observations into that tail.


## Task 4/4 — combine three sources and let the experiment disagree
For aligned normalized scores $U\in\mathbb R^{N\times D}$ and nonnegative
source weights, return $Uw/\sum_j w_j$. Validate shape, finite values and positive
total weight. A weight of zero is allowed: an ensemble can legitimately drop a source.
We compare fixed equal-weight mean and max of robust/PCA/window-IF, without a large grid.

**Before reading the table:** AP asks whether faulty times rank above normal
times, across cutoffs. State F1 balances missed faulty times and false alarms
at one chosen cutoff. Coverage is the fraction of faulty time in alarm;
normal alarm fraction is the fraction of normal time in alarm. A high score
is evidence; an **alarm** additionally requires crossing a frozen threshold.


In [ ]:
def fuse_scores(scores: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Return a weighted average of aligned normalized detector scores."""
    scores = np.asarray(scores, dtype=float)
    weights = np.asarray(weights, dtype=float)
    if scores.ndim != 2 or weights.ndim != 1 or scores.shape[1] != len(weights):
        raise ValueError("scores must be [N, D] and weights must have shape [D]")
    if not np.isfinite(scores).all() or not np.isfinite(weights).all():
        raise ValueError("scores and weights must be finite")
    if (weights < 0).any() or weights.sum() <= 0:
        raise ValueError("weights must be nonnegative with a positive sum")
    return scores @ weights / weights.sum()


In [ ]:
assert np.allclose(fuse_scores(np.array([[.2, .8]]), np.array([1., 3.])), [.65])
assert np.allclose(fuse_scores(np.array([[.2, .8]]), np.array([0., 1.])), [.8])
for invalid_values, invalid_weights in [
    (np.ones((2, 2)), np.ones(3)), (np.ones(2), np.ones(2)),
    (np.ones((2, 2)), np.array([-1., 2.])),
    (np.array([[np.nan, .5]]), np.ones(2)),
    (np.ones((2, 2)), np.array([1., np.inf])),
]:
    try:
        fuse_scores(invalid_values, invalid_weights)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid shapes, negative weights and nonfinite inputs')
try:
    fuse_scores(np.ones((2, 2)), np.zeros(2))
except ValueError:
    rejected_zero_weights = True
else:
    raise AssertionError('Zero total weight must be rejected')
core.fuse_scores = fuse_scores
SESSION.fuse()
{'weighted_mean_contract_checked': True, 'zero_weight_boundary_checked': True, 'zero_total_rejected': rejected_zero_weights}


In [ ]:
validation_report = SESSION.select_thresholds()
compact_report(validation_report)


In [ ]:
# Choose a diagnostic case on validation only; no thresholds or candidates change.
fusion_case = SESSION.validation_fusion_case()
SESSION.plot_timeline(fusion_case, ['if_raw', 'if_window', 'ensemble_mean'])


## Compare at a common alarm budget
Each validation threshold maximizes **macro per-run unadjusted state F1** while
at most **1% of pooled normal observations** may be in alarm. Mean/max require
their own thresholds. The same numeric cutoff is not the same alarm budget.

Find one interval where the window forest differs from the raw forest. Is the
difference sustained state coverage, onset timing, or recovery false alarms?
The displayed case has the largest validation mean-fusion F1 gain over window IF;
compare it with the fixed valve1/8 forecast case below instead of assuming every run benefits.
Fusion is successful only if its useful extra detections justify its extra mistakes.
A singleton can win. Do not change the 1% budget to make your preferred method win.

**A quiet policy is a real finding.** If healthy validation observations and
faulty ones saturate at the same maximum ECDF rank, a cutoff below that tie
can exceed the 1% budget. The only feasible cutoff may be above every observed
score: no alarms, F1=0. A strong AP can coexist with this unusable operating
point. Distinguish ranking quality from decision quality; do not retune the
protocol after opening held-out runs to repair the story.


In [ ]:
SESSION.plot_complementarity()


Correlation and alarm overlap are diagnostics, not proofs of ensemble value.
Similar sources can differ at an important event; different sources can add only
noise. Check actual state coverage and false alarms alongside this figure.


## What exactly does the foundation model detect?
Chronos-2-small is a pretrained forecaster. Our adapter turns it into a detector:
**past 128 steps → next 16-step median/quantiles → observe the target → residual → normal-reference rank → alarm**.

Primary forecast score: $\max_j |x_j(t)-\hat x_{j,0.5}(t)|/a_j$, using the same
normal-fit robust channel scales. We also inspect interval exceedance and a
**matched naive forecast** that repeats the last value at the same block origins.
Thus a comparison with the naive residual can reveal whether forecasting skill,
rather than merely detecting change, contributes to anomaly-state detection.


In [ ]:
# Validation only; the held-out cache is not used by this diagnostic.
FORECAST_QUALITY = forecast_quality_validation(SESSION)
display(FORECAST_QUALITY['summary'].round(4))
display(FORECAST_QUALITY['audit'])
SESSION.plot_forecast('valve1/8', 'Volume Flow RateRMS')


In [ ]:
SESSION.plot_timeline('valve1/8', ['chronos_residual', 'naive_residual', 'ensemble_mean'])


**Read the start and the middle separately.** A residual detector may fire at a
transition, then quiet down when its context contains the new regime. That can
be useful onset detection and poor whole-state coverage at the same time.
A forecast interval is not a guaranteed fault probability: its calibration may
change under distribution shift. Check whether this actually happens in the shown run.
The preceding table measures forecast error on validation only, before any
held-out reveal. It uses the same eligible targets, eight channels, fit-only
robust scales and block origins for both forecasters. The nominal 80% interval
is checked as returned, including any crossed quantiles. These are correlated
time/channel observations; the five runs do not establish general superiority.
Better forecast MAE does not imply better coverage of the faulty state.
Forest features update at each observation; the forecast refreshes every 16 steps.
We compare complete pipeline contracts and control forecasting origins with the
matched naive baseline. This is not an isolated test of neural architecture.


In [ ]:
SESSION.plot_pr('validation', ['robust', 'pca_q', 'if_raw', 'if_window', 'ensemble_mean', 'chronos_residual'])


### Read all primary metrics together
**Average Precision (AP)** summarizes precision–recall ranking without a cutoff.
The table averages AP across runs; the plotted curve pools observations.
**Unadjusted F1** evaluates every scored state label at the frozen threshold.
**State coverage** is the fraction of each real anomalous interval in alarm.
**Normal alarm fraction** is time in false alarm on observed normal seconds;
alarm episodes/hour additionally shows fragmentation.

Onset detections and delay are secondary. Missed onsets have no delay value;
their absence must not disappear behind a fast mean delay among detections.
No point adjustment fills an interval after one hit.


In [ ]:
validation_report = SESSION.metric_table("validation")

# Use a declared rule: maximize validation state F1, then AP, then lower false-alarm rate.
ranked_validation = validation_report.sort_values(
    ["macro_state_F1", "macro_AP", "pooled_normal_alarm_fraction"],
    ascending=[False, False, True],
    kind="stable",
)
selected_row = ranked_validation.iloc[0]
CHOSEN = str(selected_row["detector"])
REASON = (
    f"Selected {CHOSEN} by the validation-only rule: highest macro state F1, "
    f"then AP, with lower normal alarm fraction as the tie-breaker. "
    f"Validation F1={selected_row['macro_state_F1']:.3f}, "
    f"AP={selected_row['macro_AP']:.3f}, "
    f"coverage={selected_row['macro_state_coverage']:.3f}, "
    f"normal alarm fraction={selected_row['pooled_normal_alarm_fraction']:.3f}. "
    "Threshold selection is constrained to at most 1% pooled normal alarm time; "
    "the decision is frozen before opening held-out results."
)
compact_report(validation_report)


In [ ]:
SESSION.freeze(CHOSEN, REASON)


## Final reveal: five untouched runs
Your fitting data, CDF references, method and thresholds are now frozen. The
next cell scores the five held-out runs once. A repeated viewer reuses the saved
result. Explain a disappointing result; do not return to validation selection
or revise weights after seeing this table.


In [ ]:
holdout_report = SESSION.reveal_holdout()
compact_report(holdout_report)


In [ ]:
SESSION.plot_pr('test', ['if_raw', 'if_window', 'ensemble_mean', 'chronos_residual', 'naive_residual'])


In [ ]:
FROZEN_CHOSEN = SESSION.decision['selected']
selected_runs = core.evaluate_runs(SESSION.rows[SESSION.rows.split == 'test'], FROZEN_CHOSEN, SESSION.decision['thresholds'][FROZEN_CHOSEN]['threshold'])
selected_runs[['run_id', 'ap', 'f1', 'state_coverage_mean', 'normal_alarm_fraction', 'onsets_detected_60s', 'events', 'onset_delay_seconds_mean_detected']]


## Explain the decision, not just the leaderboard
Answer with exact runs/intervals and measured numbers.
1. Did history help the same IF? Name a benefit and a cost visible in a timeline.
2. Did classical fusion improve over the best singleton? What disagreement explains it?
3. How did Chronos compare with matched naive residuals and with classical state detectors?
4. Did the validation choice transfer? State AP, F1, coverage and false-alarm evidence.
5. What would have to be tested before using this detector on another pump/fault family?

A classical method, an ensemble or the foundation model may win. The task does
not require a predetermined ordering. This small same-rig experiment cannot
establish that one model family is generally superior.


In [ ]:
validation_rows = SESSION.rows.loc[SESSION.rows.split.eq("validation")].copy()
validation_metrics = validation_report.set_index("detector")
holdout_metrics = holdout_report.set_index("detector")

# Compare raw and history-based Isolation Forests at their own frozen thresholds.
raw_runs = core.evaluate_runs(
    validation_rows, "if_raw", SESSION.thresholds["if_raw"]["threshold"]
).set_index("run_id")
window_runs = core.evaluate_runs(
    validation_rows, "if_window", SESSION.thresholds["if_window"]["threshold"]
).set_index("run_id")
coverage_change = window_runs.state_coverage_mean - raw_runs.state_coverage_mean
history_run = str(coverage_change.abs().sort_values(ascending=False, kind="stable").index[0])
history_frame = SESSION.frames[history_run]
history_fault = history_frame.loc[history_frame.anomaly.eq(1)]
history_interval = f"{history_fault.index.min()} to {history_fault.index.max()}"

# Compare the fixed mean ensemble with the best validation singleton.
singletons = validation_report.loc[validation_report.detector.isin(["robust", "pca_q", "if_raw", "if_window"])]
best_singleton = singletons.sort_values(
    ["macro_state_F1", "macro_AP"], ascending=[False, False], kind="stable"
).iloc[0]
best_singleton_name = str(best_singleton.detector)
singleton_runs = core.evaluate_runs(
    validation_rows,
    best_singleton_name,
    SESSION.thresholds[best_singleton_name]["threshold"],
).set_index("run_id")
mean_runs = core.evaluate_runs(
    validation_rows,
    "ensemble_mean",
    SESSION.thresholds["ensemble_mean"]["threshold"],
).set_index("run_id")
fusion_change = mean_runs.state_coverage_mean - singleton_runs.state_coverage_mean
fusion_run = str(fusion_change.abs().sort_values(ascending=False, kind="stable").index[0])
fusion_frame = SESSION.frames[fusion_run]
fusion_fault = fusion_frame.loc[fusion_frame.anomaly.eq(1)]
fusion_interval = f"{fusion_fault.index.min()} to {fusion_fault.index.max()}"
fusion_rows = validation_rows.loc[validation_rows.run_id.eq(fusion_run)]
mean_alarm = fusion_rows.ensemble_mean.to_numpy() >= SESSION.thresholds["ensemble_mean"]["threshold"]
singleton_alarm = fusion_rows[best_singleton_name].to_numpy() >= SESSION.thresholds[best_singleton_name]["threshold"]
disagreement = np.flatnonzero(mean_alarm != singleton_alarm)
if len(disagreement):
    disagreement_row = fusion_rows.iloc[int(disagreement[0])]
    disagreement_note = (
        f"On {fusion_run} at {disagreement_row.datetime}, the mean and "
        f"{best_singleton_name} disagree; annotated state={int(disagreement_row.anomaly)}."
    )
else:
    disagreement_note = f"No thresholded alarm disagreement occurred on {fusion_run}."

forecast_abnormal = FORECAST_QUALITY["summary"].set_index("regime").loc["abnormal"]
chronos_row = validation_metrics.loc["chronos_residual"]
naive_row = validation_metrics.loc["naive_residual"]
classical_row = validation_metrics.loc[best_singleton_name]
selected_validation = validation_metrics.loc[CHOSEN]
selected_holdout = holdout_metrics.loc[CHOSEN]

ANSWERS = {
    "representation": (
        f"On validation, adding causal history changed IF state coverage from "
        f"{raw_runs.loc[history_run, 'state_coverage_mean']:.3f} to "
        f"{window_runs.loc[history_run, 'state_coverage_mean']:.3f} on {history_run} "
        f"during {history_interval}. Across runs, macro AP changed from "
        f"{validation_metrics.loc['if_raw', 'macro_AP']:.3f} to "
        f"{validation_metrics.loc['if_window', 'macro_AP']:.3f}. The timeline shows "
        "whether the rolling windows sustain an alarm and how their warmup affects "
        "the transition; the metric comparison alone does not establish causality."
    ),
    "fusion": (
        f"The best validation singleton by state F1 then AP was {best_singleton_name} "
        f"(F1={best_singleton['macro_state_F1']:.3f}, AP={best_singleton['macro_AP']:.3f}); "
        f"equal-weight mean had F1={validation_metrics.loc['ensemble_mean', 'macro_state_F1']:.3f}, "
        f"AP={validation_metrics.loc['ensemble_mean', 'macro_AP']:.3f}. On {fusion_run} "
        f"during {fusion_interval}, mean-minus-singleton coverage was "
        f"{fusion_change.loc[fusion_run]:+.3f}. {disagreement_note}"
    ),
    "forecasting": (
        f"On validation, Chronos residual state AP/F1 were "
        f"{chronos_row['macro_AP']:.3f}/{chronos_row['macro_state_F1']:.3f}; "
        f"matched naive residual values were {naive_row['macro_AP']:.3f}/"
        f"{naive_row['macro_state_F1']:.3f}; {best_singleton_name} had "
        f"{classical_row['macro_AP']:.3f}/{classical_row['macro_state_F1']:.3f}. "
        f"For abnormal forecast targets, normalized MAE was "
        f"{forecast_abnormal['chronos_nmae']:.3f} for Chronos and "
        f"{forecast_abnormal['persistence_nmae']:.3f} for persistence. Forecast "
        "accuracy and full-state alarm coverage are different objectives; these "
        "small validation runs do not establish general model superiority."
    ),
    "transfer": (
        f"The frozen validation choice was {CHOSEN}: AP={selected_validation['macro_AP']:.3f}, "
        f"F1={selected_validation['macro_state_F1']:.3f}, "
        f"coverage={selected_validation['macro_state_coverage']:.3f}, "
        f"normal alarm fraction={selected_validation['pooled_normal_alarm_fraction']:.3f}. "
        f"On the five held-out runs it scored AP={selected_holdout['macro_AP']:.3f}, "
        f"F1={selected_holdout['macro_state_F1']:.3f}, "
        f"coverage={selected_holdout['macro_state_coverage']:.3f}, "
        f"normal alarm fraction={selected_holdout['pooled_normal_alarm_fraction']:.3f}. "
        "This is a single fixed split transfer check, not a guarantee for another pump."
    ),
    "scope": (
        "Before deployment on another pump or fault family, collect independent runs, "
        "fit calibration and thresholds only on healthy data from the intended setting, "
        "and validate the alarm budget, event coverage, delay, and false-alarm episodes "
        "with a preregistered split. Monitor sensor drift and require operator review; "
        "the present experiment covers one rig and a small set of runs."
    ),
}
ANSWERS


In [ ]:
SESSION.save_submission(ANSWERS)


## Optional: run one live foundation forecast
This extension is **off by default** and is not needed for the assessed route.
Enabling it downloads approximately 112 MB of weights and adds Torch/Chronos
dependencies. Use a fresh runtime if installation conflicts arise. CPU is supported;
optional GPU changes speed, not the experiment definition. The recorded cache
remains identified separately from this new one-block execution.


In [ ]:
RUN_LIVE_CHRONOS = False
if RUN_LIVE_CHRONOS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'chronos-forecasting==2.3.2', 'transformers==4.57.1'])
    import torch
    from chronos import Chronos2Pipeline
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    live_pipeline = Chronos2Pipeline.from_pretrained('autogluon/chronos-2-small', revision='ddec01313e50b6bc58ebaa92ede81bc24a3d9f9a', device_map=device, torch_dtype=torch.float32)
    live_frame = SESSION.frames['valve1/8'].iloc[:128]
    live_input = live_frame[CHANNELS].to_numpy(dtype=np.float32).T
    live_output = live_pipeline.predict([{'target': live_input}], prediction_length=16, batch_size=64, context_length=128, cross_learning=False)
    print({'new_execution': True, 'device': device, 'prediction_shape': tuple(live_output[0].shape), 'context': 128, 'horizon': 16})
else:
    print('Optional live model execution is off. The required route used the labelled recorded forecast cache.')


## Submit and attribute
Save your completed notebook and the three small files listed above. Do not submit
model caches or credentials. Be prepared to explain or modify every task function.

**Checks:** four task contracts pass; fresh top-to-bottom run works; split identities
stay fixed; no future leakage; frozen decision precedes heldout reveal; five answers
cite actual evidence. Teacher solutions remain in the separate private notebook.

**Sources:** [SKAB repository](https://github.com/waico/SKAB), Katser & Kozitsin (2020),
[dataset DOI](https://doi.org/10.34740/KAGGLE/DSV/1693952), GPL-3.0;
[Chronos-2-small model](https://huggingface.co/autogluon/chronos-2-small),
[Chronos implementation](https://github.com/amazon-science/chronos-forecasting).
The exact dataset/model revisions and recorded execution environment are included
in the verified package. No claim of general foundation-model superiority or inferiority
follows from this one laboratory.
